#Demonstration: Adapting a Policy Bot with QLoRA

# Scenario

* PolicyDoc Micro — a small insurer needs short, domain-specific answers (claims deadlines, required documents). They adapt a tiny LLM efficiently and check that tuned replies beat the baseline.

# Objectives

* Prepare a tiny policy dataset
* Apply QLoRA
* Run a tiny fine-tuning loop
* Evaluate tuned vs baseline with a simple metric

A small insurance company, PolicyDoc Micro, needs an AI assistant capable of giving short, precise, domain-specific answers related to their policies. Customers often ask:

“What documents do I need for a claim?”

“How long do I have to report an accident?”

“What is the waiting period?”

Because the insurer is small, they want a compact, low-cost model that can run locally or on inexpensive hardware. Instead of using large hosted models, they plan to adapt a tiny LLM using QLoRA, a very efficient fine-tuning method that uses quantization + LoRA adapters.

Their goal is to verify that the fine-tuned model produces better answers than the original baseline model.

1️⃣ Prepare a tiny policy dataset

You will build a small training set containing:

policy-related questions (“instruction”)

accurate, domain-specific answers (“response”)

This dataset will be small — maybe 20–40 examples — but enough to shift the behavior of a compact LLM.

2️⃣ Apply QLoRA (Quantized LoRA)

Instead of normal LoRA, we use QLoRA, which:

Loads the base model in 4-bit quantized format → saves memory

Trains only LoRA adapter layers → low compute

Works great on CPUs and small GPUs

This makes it perfect for startups with limited resources.

3️⃣ Run a tiny fine-tuning loop

You will:

Load the quantized model

Attach LoRA adapters

Tokenize the dataset

Run a small training loop (2–5 epochs)

Update only adapter weights

This is enough for the model to learn policy-specific phrasing and rules.

4️⃣ Evaluate tuned vs. baseline

After training:

Generate answers using baseline model

Generate answers using fine-tuned model

Compare them using a simple metric, such as:

BLEU

ROUGE-L

Embedding similarity

Or a simple heuristic (“closer to ground truth?”)

This demonstrates the practical improvements gained from fine-tuning.

# Install Libraries

In [ ]:
!pip -q install transformers peft


# Setup

This code sets up the model-loading stage for a QLoRA-based fine-tuning experiment. It begins by loading the tokenizer for a very small causal language model (sshleifer/tiny-gpt2), ensuring a valid padding token is available to avoid batching issues. The script then determines whether a GPU is available, because QLoRA requires CUDA to load models in 4-bit precision. The function try_load_4bit() attempts to import the BitsAndBytes library (bitsandbytes), which provides efficient 4-bit quantization, and then tries to load the model using load_in_4bit=True with automatic device mapping. If this succeeds, the model loads in low-memory 4-bit mode, enabling QLoRA; otherwise, the function returns a failure signal. Based on this outcome, the script either uses the quantized model (QLoRA mode) or falls back to a standard full-precision (FP32) model on CPU/GPU for normal LoRA training. Finally, it prints whether the environment successfully enabled QLoRA or had to fall back to regular LoRA. This makes the notebook robust across environments, gracefully adapting to whether quantization tools and GPUs are available.

In [ ]:
import torch, importlib
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_name = "sshleifer/tiny-gpt2"
tok = AutoTokenizer.from_pretrained(model_name)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"

def try_load_4bit():
    if not torch.cuda.is_available():
        return None, False
    try:
        import bitsandbytes as bnb  # noqa: F401
        m = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit=True, device_map="auto")
        return m, True
    except Exception:
        return None, False

base_4bit, is_qlora = try_load_4bit()
base = base_4bit if is_qlora else AutoModelForCausalLM.from_pretrained(model_name).to(device)
print("Using QLoRA 4-bit:" if is_qlora else "Fallback to LoRA (FP32/CPU or CUDA):", not not is_qlora)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Fallback to LoRA (FP32/CPU or CUDA): False


#Domain Data

This block constructs a small, domain-specific training dataset for the insurance QLoRA fine-tuning task. Each training pair consists of a short policy-related question and its ideal answer, such as the claim filing deadline or required documents. These pairs are converted into a consistent instruction–response format using the fmt() helper function, producing training strings that the model can learn from in a causal language-modeling setup. The list train_texts is tokenized using the same tokenizer as the base model. If QLoRA is not active (i.e., the model could not be loaded in 4-bit mode), the input tensors are moved onto the appropriate device (CPU or GPU). The labels are created by cloning the input IDs, following the standard causal LM training approach where the model learns to predict the next token in the combined instruction–response text. This block prepares the training data for both LoRA and QLoRA fine-tuning in the subsequent steps.

In [ ]:
pairs = [
    ("What is the claim filing deadline", "Claims must be filed within 30 days of the incident."),
    ("What documents are needed for a claim", "Provide photo ID, proof of ownership, and incident report."),
    ("Can I add a new driver", "Yes. Add the driver in the portal. Coverage starts after confirmation email."),
    ("How do I change my address", "Update your address in the portal. Changes take effect within 24 hours."),
]
eval_prompts = [
    "What is the claim filing deadline",
    "What documents are needed for a claim",
]

def fmt(p, r):
    return f"Instruction: {p}\nResponse: {r}"
train_texts = [fmt(p,r) for p,r in pairs]
inputs = tok(train_texts, padding=True, return_tensors="pt")
if not is_qlora:
    inputs = {k:v.to(device) for k,v in inputs.items()}
labels = inputs["input_ids"].clone()


#QLoRA Adapters

This block configures and attaches LoRA (or QLoRA) adapters to the base model so that only a small set of lightweight parameters will be trained during fine-tuning. The targets list specifies the internal attention projection layers (c_attn and c_proj) of Tiny-GPT2 where the LoRA adapters should be inserted—these layers are the most influential in shaping the model’s responses. The LoraConfig defines the adapter behavior: r=4 sets the low-rank dimension, lora_alpha=8 scales the updates, and lora_dropout=0.05 provides regularization. Using get_peft_model(), the chosen adapters are injected into the base model, producing a parameter-efficient version ready for fine-tuning. If QLoRA is not available (i.e., the model is not loaded in 4-bit quantized mode), the adapted model is moved onto the appropriate device (CPU or GPU). Finally, model.train() places the model in training mode, preparing it for the subsequent fine-tuning loop.

In [ ]:
targets = ["c_attn","c_proj"]  # tiny-gpt2 attn/proj names
cfg = LoraConfig(r=4, lora_alpha=8, lora_dropout=0.05, target_modules=targets)
model = get_peft_model(base, cfg)
if not is_qlora:
    model.to(device)
model.train()


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


PeftModel(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 2)
        (wpe): Embedding(1024, 2)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-1): 2 x GPT2Block(
            (ln_1): LayerNorm((2,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=6, nx=2)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=6, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector)

#Training

This block performs a minimal fine-tuning step on the LoRA/QLoRA-adapted Tiny-GPT2 model using the small insurance-domain training dataset. An AdamW optimizer is created with a relatively high learning rate (1e-3), which is acceptable because only a very small number of LoRA adapter parameters are being trained. The loop runs for just three epochs—sufficient for a quick demonstration. In each epoch, gradients are reset, the model computes the loss by predicting the next token in the instruction–response text, and backpropagation updates only the LoRA adapter weights. Because the base model remains frozen, training is extremely fast and lightweight. The loss value is printed after each epoch, allowing you to confirm that the model is successfully learning the domain-specific response patterns. This completes the fine-tuning stage of the QLoRA pipeline.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
for e in range(3):
    opt.zero_grad()
    out = model(**inputs, labels=labels)
    loss = out.loss
    loss.backward()
    opt.step()
    print(f"Epoch {e+1} Loss: {loss.item():.4f}")


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1 Loss: 10.8244
Epoch 2 Loss: 10.8224


model.safetensors:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Epoch 3 Loss: 10.8238


#Evaluate vs Baseline

This section evaluates how much the fine-tuned LoRA/QLoRA model improves over the original baseline model using a simple automatic metric. The generate() function formats a prompt in the same instruction–response style as the training data, runs it through the model, and extracts only the generated response text. The evaluation metric f1() computes an approximate token-overlap F1 score between the model’s predicted answer and the ground-truth response, treating each answer as a set of lowercase words. This gives a lightweight, interpretable measure of similarity suitable for tiny models and small datasets.

A small gold reference set is defined for two evaluation prompts. To ensure fairness, the code reloads a fresh instance of the untuned base model as baseline, while the fine-tuned model with LoRA adapters is used as model. The helper avg_f1() generates answers for each prompt and computes the average F1 score. The script prints both the baseline and tuned F1 scores, showing whether fine-tuning improved alignment with the ground-truth answers. Finally, for each evaluation prompt, the code prints both models’ actual responses side-by-side, making it easy to qualitatively compare the improvements in clarity, accuracy, and domain specificity. This concludes the end-to-end QLoRA fine-tuning and evaluation pipeline.

In [ ]:
def generate(m, prompt):
    x = tok(f"Instruction: {prompt}\nResponse:", return_tensors="pt")
    if not is_qlora: x = {k:v.to(device) for k,v in x.items()}
    y = m.generate(**x, max_new_tokens=32, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(y[0], skip_special_tokens=True).split("Response:",1)[-1].strip()

def f1(a,b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    if not sa or not sb: return 0.0
    tp = len(sa & sb); p = tp/max(1,len(sb)); r = tp/max(1,len(sa))
    return 0.0 if p+r==0 else 2*p*r/(p+r)

gold = {
    "What is the claim filing deadline": "Claims must be filed within 30 days of the incident.",
    "What documents are needed for a claim": "Provide photo ID, proof of ownership, and incident report.",
}

# rebuild a plain baseline copy for fair compare
baseline = AutoModelForCausalLM.from_pretrained(model_name)
if not is_qlora: baseline.to(device)

def avg_f1(model_ref):
    scores = []
    for p in eval_prompts:
        pred = generate(model_ref, p)
        scores.append(f1(gold[p], pred))
    return sum(scores)/len(scores)

print("Baseline F1:", round(avg_f1(baseline),3))
print("Tuned F1:", round(avg_f1(model),3))

for p in eval_prompts:
    print("\nPrompt:", p)
    print("Baseline:", generate(baseline, p))
    print("Tuned   :", generate(model, p))


Baseline F1: 0.0
Tuned F1: 0.0

Prompt: What is the claim filing deadline
Baseline: stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs
Tuned   : stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs

Prompt: What documents are needed for a claim
Baseline: stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs
Tuned   : stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs